## Group Number 51 :-



1.   AGRAWAL SHRIYA RAVINDRA (2023ac05857)
2.   BRIJESH PRAVINCHANDRA SONI (2023ac05048)
3.   C S PRADEEP (2023ac05118)
4.   CHAUDHARI ATHARV HEMANTKUMAR (2023ac05081)
5.   NATU CHINMAY VIVEK (2023ac05116)



In [7]:
%%capture
!pip install faiss-cpu rank-bm25

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd

pd.read_csv("/content/drive/MyDrive/Conversational AI/Processing Files/qna_data/nestle_qna_pairs.csv").head()

,Question,Answer
0,Q: What were Nestlé's sales in 2023?,A: Nestlé's sales in 2023 were CHF 92 998 mill...
1,Q: What was the financial income reported for ...,A: Financial income for 2024 was CHF 358 million.
2,Q: What were Nestlé's research and development...,A: Research and development costs were CHF (1 ...
3,Q: What was the profit for the year attributab...,A: The profit for the year attributable to sha...
4,Q: What was Nestlé's other revenue in 2023?,A: Nestlé's other revenue in 2023 was CHF 353 ...


Data Processing and chunking the data

In [ ]:
# ==============================
# Step 2.1: Data Processing
# ==============================
import os
import shutil
import pandas as pd
from transformers import AutoTokenizer

drive_qna_path = "/content/drive/MyDrive/Conversational AI/Processing Files/qna_data/nestle_qna_pairs.csv"
local_dir = "/content/qna_data"
local_qna_path = os.path.join(local_dir, "nestle_qna_pairs.csv")

os.makedirs(local_dir, exist_ok=True)

if not os.path.exists(local_qna_path):
    shutil.copy2(drive_qna_path, local_qna_path)
    print(f"✅ Copied QnA CSV to {local_qna_path}")
else:
    print(f"ℹ️ QnA CSV already exists locally at {local_qna_path}")

df = pd.read_csv(local_qna_path)
df = df.dropna(subset=["Question", "Answer"])
print(f"✅ Loaded {len(df)} QnA pairs")

tokenizer = AutoTokenizer.from_pretrained("gpt2")

def chunk_text(text, max_tokens=100):
    """Split text into chunks with <= max_tokens using tokenizer."""
    tokens = tokenizer.encode(text)
    for i in range(0, len(tokens), max_tokens):
        chunk = tokens[i:i+max_tokens]
        yield tokenizer.decode(chunk)

chunks_100, chunks_400 = [], []

for idx, row in df.iterrows():
    q = str(row["Question"]).strip()
    a = str(row["Answer"]).strip()
    full_text = f"Q: {q}\nA: {a}"

    for c_idx, chunk in enumerate(chunk_text(full_text, max_tokens=100)):
        chunks_100.append({
            "id": f"qna_{idx}_100_{c_idx}",
            "source": "nestle_qna_pairs.csv",
            "chunk_size": 100,
            "text": chunk
        })

    for c_idx, chunk in enumerate(chunk_text(full_text, max_tokens=400)):
        chunks_400.append({
            "id": f"qna_{idx}_400_{c_idx}",
            "source": "nestle_qna_pairs.csv",
            "chunk_size": 400,
            "text": chunk
        })

print(f"✅ Generated {len(chunks_100)} small chunks (~100 tokens)")
print(f"✅ Generated {len(chunks_400)} large chunks (~400 tokens)")

proc_dir = "/content/drive/MyDrive/Hackathons Stuff/Financial Q&A Chat Bot/processed_chunks"
os.makedirs(proc_dir, exist_ok=True)

pd.DataFrame(chunks_100).to_csv(os.path.join(proc_dir, "chunks_100.csv"), index=False)
pd.DataFrame(chunks_400).to_csv(os.path.join(proc_dir, "chunks_400.csv"), index=False)

print(f"📂 Chunks saved in {proc_dir}")

ℹ️ QnA CSV already exists locally at /content/qna_data/nestle_qna_pairs.csv
✅ Loaded 68 QnA pairs


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

✅ Generated 68 small chunks (~100 tokens)
✅ Generated 68 large chunks (~400 tokens)
📂 Chunks saved in /content/drive/MyDrive/Hackathons Stuff/Financial Q&A Chat Bot/processed_chunks


code to create faiss index and Dense retrieval (vector similarity).
Sparse retrieval (BM25).

In [10]:
import os
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import pickle

drive_base = "/content/drive/MyDrive/Conversational AI/Processing Files"
qna_path = os.path.join(drive_base, "qna_data/nestle_qna_pairs.csv")
index_dir = os.path.join(drive_base, "indexes")
embeddings_dir = os.path.join(index_dir, "dense_embeddings")
os.makedirs(index_dir, exist_ok=True)
os.makedirs(embeddings_dir, exist_ok=True)

qna_df = pd.read_csv(qna_path)
qna_df["Question"] = qna_df["Question"].str.replace(r"^Q:\s*", "", regex=True)
qna_df["Answer"] = qna_df["Answer"].str.replace(r"^A:\s*", "", regex=True)
qna_df["document"] = qna_df["Question"] + " " + qna_df["Answer"]
print(f"✅ Loaded {len(qna_df)} Q&A pairs from {qna_path}")

embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = embed_model.encode(
    qna_df["document"].tolist(),
    batch_size=32,
    show_progress_bar=True
).astype("float32")

for i, embedding in enumerate(embeddings):
    embedding_file_path = os.path.join(embeddings_dir, f"embedding_{i}.npy")
    np.save(embedding_file_path, embedding)

dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

faiss_index_path = os.path.join(index_dir, "nestle_qna.index")
faiss.write_index(index, faiss_index_path)
print(f"✅ FAISS index stored at {faiss_index_path}")

meta_path = os.path.join(index_dir, "metadata.pkl")
with open(meta_path, "wb") as f:
    pickle.dump(qna_df.to_dict(orient="records"), f)
print(f"✅ Metadata saved at {meta_path}")

tokenized_corpus = [doc.split(" ") for doc in qna_df["document"].tolist()]
bm25 = BM25Okapi(tokenized_corpus)

bm25_path = os.path.join(index_dir, "bm25_index.pkl")
with open(bm25_path, "wb") as f:
    pickle.dump(bm25, f)
print(f"✅ BM25 index saved at {bm25_path}")

✅ Loaded 68 Q&A pairs from /content/drive/MyDrive/Conversational AI/Processing Files/qna_data/nestle_qna_pairs.csv


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

✅ FAISS index stored at /content/drive/MyDrive/Conversational AI/Processing Files/indexes/nestle_qna.index
✅ Metadata saved at /content/drive/MyDrive/Conversational AI/Processing Files/indexes/metadata.pkl
✅ BM25 index saved at /content/drive/MyDrive/Conversational AI/Processing Files/indexes/bm25_index.pkl


Hybrid Retrieval Pipeline

In [11]:
import os
import re
import faiss
import pickle
import numpy as np
import pandas as pd
from typing import Dict, Any, List
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

BASE_DIR = "/content/drive/MyDrive/Conversational AI/Processing Files"
INDEX_DIR = os.path.join(BASE_DIR, "indexes")
FAISS_FILE = os.path.join(INDEX_DIR, "nestle_qna.index")
BM25_FILE = os.path.join(INDEX_DIR, "bm25_index.pkl")
META_FILE = os.path.join(INDEX_DIR, "metadata.pkl")

faiss_index = faiss.read_index(FAISS_FILE)
with open(BM25_FILE, "rb") as f:
    bm25_index = pickle.load(f)
with open(META_FILE, "rb") as f:
    qna_records = pickle.load(f)

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def normalize_text(text: str) -> List[str]:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    tokens = text.split()
    return [tok for tok in tokens if tok not in ENGLISH_STOP_WORDS]

def retrieve_dense(query: str, k: int = 5) -> List[Dict[str, Any]]:
    q_emb = embedder.encode([query]).astype("float32")
    D, I = faiss_index.search(q_emb, k)
    hits = []
    for rank, (idx, dist) in enumerate(zip(I[0], D[0])):
        rec = qna_records[idx]
        hits.append({
            "id": str(idx),
            "text": rec["document"],
            "question": rec["Question"],
            "answer": rec["Answer"],
            "dense": 1.0 / (1.0 + dist)
        })
    return hits

def retrieve_sparse(query: str, k: int = 5) -> List[Dict[str, Any]]:
    tokens = normalize_text(query)
    scores = bm25_index.get_scores(tokens)
    ranked = np.argsort(scores)[::-1][:k]
    results = []
    for idx in ranked:
        rec = qna_records[idx]
        results.append({
            "id": str(idx),
            "text": rec["document"],
            "question": rec["Question"],
            "answer": rec["Answer"],
            "sparse": float(scores[idx])
        })
    return results

def retrieve_hybrid(query: str, k: int = 5, weight_dense: float = 0.6) -> List[Dict[str, Any]]:
    dense_hits = {h["id"]: h for h in retrieve_dense(query, k)}
    sparse_hits = {h["id"]: h for h in retrieve_sparse(query, k)}
    combined: Dict[str, Dict[str, Any]] = {}
    for hit in [*dense_hits.values(), *sparse_hits.values()]:
        if hit["id"] not in combined:
            combined[hit["id"]] = hit.copy()
        else:
            combined[hit["id"]].update(hit)
    for doc in combined.values():
        d = doc.get("dense", 0.0)
        s = doc.get("sparse", 0.0)
        doc["hybrid"] = weight_dense * d + (1 - weight_dense) * s
    ranked = sorted(combined.values(), key=lambda x: x["hybrid"], reverse=True)[:k]
    return ranked

query = "What did the 2024 other revenue of primarily sale of outsourced transportation services include?"
top_docs = retrieve_hybrid(query, k=5)

for d in top_docs:
    print(f"\n🔹 ID: {d['id']} | Hybrid score: {d['hybrid']:.4f}")
    print(f"Q: {d['question']}")
    print(f"A: {d['answer']}")
    print(f"Snippet: {d['text'][:200]}...")


🔹 ID: 44 | Hybrid score: 12.8484
Q: What did the 2024 other revenue of primarily sale of outsourced transportation services include?
A: The 2024 other revenue of primarily sale of outsourced transportation services included the salaries and wages of drivers, warehouse employees and customer service staff.
Snippet: What did the 2024 other revenue of primarily sale of outsourced transportation services include? The 2024 other revenue of primarily sale of outsourced transportation services included the salaries an...

🔹 ID: 4 | Hybrid score: 1.7980
Q: What was Nestlé's other revenue in 2023?
A: Nestlé's other revenue in 2023 was CHF 353 million.
Snippet: What was Nestlé's other revenue in 2023? Nestlé's other revenue in 2023 was CHF 353 million....

🔹 ID: 65 | Hybrid score: 1.2580
Q: What was the total number of factories and countries of sale the Group has?
A: The Group has factories in 75 countries and sales in 185 countries.
Snippet: What was the total number of factories and countrie

Advanced RAG: Cross-Encoder Re-ranking (Multi-Stage Retrieval)
--------------------------------------------------------------
Stage 1: use your existing hybrid/dense/sparse retrievers to get candidates.

Stage 2: re-rank those candidates with a cross-encoder for precise scoring.

stage 3: genrate response

stage 4: guardrail

In [23]:
from __future__ import annotations

from typing import List, Dict, Any, Tuple
import os
import re
import pickle
import numpy as np
import pandas as pd
import faiss
import torch
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
)

# =============================================================================
# Paths (existing indexes)
# =============================================================================

INDEX_DIR = "/content/drive/MyDrive/Conversational AI/Processing Files/indexes"
FAISS_INDEX_PATH = os.path.join(INDEX_DIR, "nestle_qna.index")
BM25_INDEX_PATH = os.path.join(INDEX_DIR, "bm25_index.pkl")
META_PATH = os.path.join(INDEX_DIR, "metadata.pkl")
CSV_FALLBACK = "/content/drive/MyDrive/Conversational AI/Processing Files/qna_data/nestle_qna_pairs.csv"

# =============================================================================
# Loading utilities
# =============================================================================

def load_metadata(meta_path: str, csv_fallback: str) -> List[Dict[str, Any]]:
    """
    Load metadata records aligned with FAISS/BM25 indexes.

    Parameters
    ----------
    meta_path : str
        Path to a pickle file that contains a list of dict records.
    csv_fallback : str
        CSV path used if the pickle does not exist. Must contain
        'Question' and 'Answer' columns.

    Returns
    -------
    List[Dict[str, Any]]
        Records with at least 'document' text per item.
    """
    if os.path.isfile(meta_path):
        with open(meta_path, "rb") as f:
            records = pickle.load(f)
    else:
        df = pd.read_csv(csv_fallback)
        df["Question"] = df["Question"].str.replace(r"^Q:\s*", "", regex=True).fillna("")
        df["Answer"] = df["Answer"].str.replace(r"^A:\s*", "", regex=True).fillna("")
        df["document"] = (df["Question"].astype(str) + " " + df["Answer"].astype(str)).str.strip()
        records = df.to_dict(orient="records")
    return records


def load_faiss_index(index_path: str) -> faiss.Index:
    """
    Load a FAISS index from disk.

    Parameters
    ----------
    index_path : str
        Path to the FAISS index file.

    Returns
    -------
    faiss.Index
        Loaded FAISS index instance.
    """
    if not os.path.isfile(index_path):
        raise FileNotFoundError(f"FAISS index not found at: {index_path}")
    return faiss.read_index(index_path)


def load_bm25_index(pickle_path: str):
    """
    Load a pickled BM25Okapi index.

    Parameters
    ----------
    pickle_path : str
        Path to the BM25 pickle file.

    Returns
    -------
    Any
        BM25 index object with .get_scores(tokens).
    """
    if not os.path.isfile(pickle_path):
        raise FileNotFoundError(f"BM25 index not found at: {pickle_path}")
    with open(pickle_path, "rb") as f:
        return pickle.load(f)

# =============================================================================
# Retrieval: dense (FAISS) + sparse (BM25)
# =============================================================================

def _normalize_scores(scores: List[float]) -> List[float]:
    """
    Normalize a list of scores into [0, 1] range with numerical stability.

    Parameters
    ----------
    scores : List[float]
        Raw scores.

    Returns
    -------
    List[float]
        Min–max normalized scores (0 if constant).
    """
    if not scores:
        return scores
    s = np.asarray(scores, dtype=np.float32)
    lo, hi = float(np.min(s)), float(np.max(s))
    if hi <= lo:
        return [0.0] * len(scores)
    return ((s - lo) / (hi - lo)).tolist()


def retrieve_dense_faiss(
    query: str,
    index: faiss.Index,
    embedder: SentenceTransformer,
    records: List[Dict[str, Any]],
    top_k: int = 20,
) -> List[Dict[str, Any]]:
    """
    Perform dense retrieval using FAISS (IndexFlatL2).

    Parameters
    ----------
    query : str
        User query.
    index : faiss.Index
        Loaded FAISS index (built with all-MiniLM-L6-v2 embeddings).
    embedder : SentenceTransformer
        The same embedding model used to build the index.
    records : List[Dict[str, Any]]
        Metadata records aligned to the index order.
    top_k : int
        Number of candidates to return.

    Returns
    -------
    List[Dict[str, Any]]
        Dense candidates with 'id', 'text', and 'dense_score'.
    """
    q_vec = embedder.encode([query]).astype("float32")
    distances, idxs = index.search(q_vec, top_k)
    idxs = idxs[0].tolist()
    dists = distances[0].tolist()

    # Convert L2 distance → similarity-like score.
    sims = [1.0 / (1.0 + float(d)) for d in dists]
    sims = _normalize_scores(sims)

    results: List[Dict[str, Any]] = []
    for i, sim in zip(idxs, sims):
        if 0 <= i < len(records):
            rec = records[i]
            text = rec.get("document") or (
                (rec.get("Question", "") + " " + rec.get("Answer", "")).strip()
            )
            results.append(
                {
                    "id": i,
                    "text": text,
                    "dense_score": float(sim),
                    "source": rec.get("source"),
                }
            )
    return results


def _simple_tokenize(s: str) -> List[str]:
    """
    Light tokenization for BM25 queries (to match simple corpus tokenization).

    Parameters
    ----------
    s : str
        Input text.

    Returns
    -------
    List[str]
        Lowercased, whitespace-split tokens with punctuation removed.
    """
    s = s.lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    return [t for t in s.split() if t]


def retrieve_sparse_bm25(
    query: str,
    bm25,
    records: List[Dict[str, Any]],
    top_k: int = 20,
) -> List[Dict[str, Any]]:
    """
    Perform sparse retrieval using a prebuilt BM25 index.

    Parameters
    ----------
    query : str
        User query.
    bm25 : Any
        BM25 index object with .get_scores(tokens).
    records : List[Dict[str, Any]]
        Metadata records aligned with the BM25 corpus order.
    top_k : int
        Number of candidates to return.

    Returns
    -------
    List[Dict[str, Any]]
        Sparse candidates with 'id', 'text', and 'sparse_score'.
    """
    tokens = _simple_tokenize(query)
    scores = bm25.get_scores(tokens)  # type: ignore[attr-defined]
    if isinstance(scores, list):
        scores = np.array(scores, dtype=np.float32)
    order = np.argsort(-scores)[:top_k]

    raw = []
    for i in order.tolist():
        if 0 <= i < len(records):
            rec = records[i]
            text = rec.get("document") or (
                (rec.get("Question", "") + " " + rec.get("Answer", "")).strip()
            )
            raw.append({"id": i, "text": text, "sparse_score": float(scores[i])})

    # Normalize sparse scores to [0, 1] within this candidate set.
    norm = _normalize_scores([r["sparse_score"] for r in raw])
    for r, ns in zip(raw, norm):
        r["sparse_score"] = float(ns)
    return raw


def retrieve_hybrid(
    query: str,
    index: faiss.Index,
    bm25,
    embedder: SentenceTransformer,
    records: List[Dict[str, Any]],
    top_k: int = 20,
    weight_dense: float = 0.6,
) -> List[Dict[str, Any]]:
    """
    Fuse dense (FAISS) and sparse (BM25) results by weighted score.

    Parameters
    ----------
    query : str
        User query.
    index : faiss.Index
        FAISS index instance.
    bm25 : Any
        BM25 index instance.
    embedder : SentenceTransformer
        Query encoder (same model as used for FAISS).
    records : List[Dict[str, Any]]
        Metadata records aligned to both indexes.
    top_k : int
        Number of final candidates to return.
    weight_dense : float
        Weight for dense score in the fusion (0..1).

    Returns
    -------
    List[Dict[str, Any]]
        Hybrid top-k results with 'hybrid_score'.
    """
    dense_hits = retrieve_dense_faiss(query, index, embedder, records, top_k=top_k)
    sparse_hits = retrieve_sparse_bm25(query, bm25, records, top_k=top_k)

    by_id: Dict[int, Dict[str, Any]] = {}
    for h in dense_hits:
        by_id[h["id"]] = dict(h)
    for h in sparse_hits:
        if h["id"] in by_id:
            by_id[h["id"]]["sparse_score"] = h["sparse_score"]
        else:
            by_id[h["id"]] = dict(h)

    for v in by_id.values():
        ds = float(v.get("dense_score", 0.0))
        ss = float(v.get("sparse_score", 0.0))
        v["hybrid_score"] = weight_dense * ds + (1.0 - weight_dense) * ss

    ranked = sorted(by_id.values(), key=lambda x: x["hybrid_score"], reverse=True)
    return ranked[:top_k]

# =============================================================================
# Cross-encoder re-ranking
# =============================================================================

def build_reranker(
    model_id: str = "cross-encoder/ms-marco-MiniLM-L-6-v2",
) -> Tuple[AutoTokenizer, AutoModelForSequenceClassification, torch.device]:
    """
    Initialize cross-encoder reranker on the best available device.

    Parameters
    ----------
    model_id : str
        Hugging Face model identifier.

    Returns
    -------
    Tuple[AutoTokenizer, AutoModelForSequenceClassification, torch.device]
        Tokenizer, model, and device.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tok = AutoTokenizer.from_pretrained(model_id)
    mdl = AutoModelForSequenceClassification.from_pretrained(model_id)
    mdl.to(device)
    mdl.eval()
    return tok, mdl, device


def _score_batches(
    query: str,
    passages: List[str],
    tokenizer: AutoTokenizer,
    model: AutoModelForSequenceClassification,
    device: torch.device,
    batch_size: int = 16,
    max_length: int = 512,
) -> List[float]:
    """
    Score (query, passage) pairs in batches using a cross-encoder.

    Parameters
    ----------
    query : str
        User question.
    passages : List[str]
        Candidate texts to score.
    tokenizer : AutoTokenizer
        Cross-encoder tokenizer.
    model : AutoModelForSequenceClassification
        Cross-encoder model.
    device : torch.device
        Compute device.
    batch_size : int
        Batch size for forward passes.
    max_length : int
        Truncation length for the tokenizer.

    Returns
    -------
    List[float]
        Float scores per passage.
    """
    scores: List[float] = []
    total = len(passages)

    for start in range(0, total, batch_size):
        end = start + batch_size
        batch = passages[start:end]
        enc = tokenizer(
            [query] * len(batch),
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        ).to(device)

        with torch.inference_mode():
            logits = model(**enc).logits
        arr = logits.squeeze(-1).detach().float().cpu().tolist()
        if isinstance(arr, float):
            arr = [arr]
        scores.extend(arr)

    return scores


def rerank_candidates(
    query: str,
    candidates: List[Dict[str, Any]],
    tokenizer: AutoTokenizer,
    model: AutoModelForSequenceClassification,
    device: torch.device,
    top_k: int = 5,
    batch_size: int = 16,
) -> List[Dict[str, Any]]:
    """
    Re-rank hybrid candidates using a cross-encoder.

    Parameters
    ----------
    query : str
        User question.
    candidates : List[Dict[str, Any]]
        Hybrid results with 'text'.
    tokenizer : AutoTokenizer
        Cross-encoder tokenizer.
    model : AutoModelForSequenceClassification
        Cross-encoder model.
    device : torch.device
        Compute device.
    top_k : int
        Number of passages to keep.
    batch_size : int
        Batch size for scoring.

    Returns
    -------
    List[Dict[str, Any]]
        Top-k candidates with 'rerank_score'.
    """
    if not candidates:
        return []

    texts = [c["text"] for c in candidates]
    scores = _score_batches(query, texts, tokenizer, model, device, batch_size=batch_size)
    enriched = []
    for c, s in zip(candidates, scores):
        item = dict(c)
        item["rerank_score"] = float(s)
        enriched.append(item)
    enriched.sort(key=lambda x: x["rerank_score"], reverse=True)
    return enriched[:top_k]

# =============================================================================
# Generator (DistilGPT2) with strict prompt
# =============================================================================

def load_generator(model_name: str = "distilgpt2") -> Tuple[AutoTokenizer, AutoModelForCausalLM, torch.device]:
    """
    Load a compact causal LM for answer synthesis.

    Parameters
    ----------
    model_name : str
        Hugging Face model id.

    Returns
    -------
    Tuple[AutoTokenizer, AutoModelForCausalLM, torch.device]
        Tokenizer, model, device.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tok = AutoTokenizer.from_pretrained(model_name)
    mdl = AutoModelForCausalLM.from_pretrained(model_name)
    mdl.to(device)
    mdl.eval()
    return tok, mdl, device


def _build_answer_prompt(query: str, context: str) -> str:
    """
    Build a constrained prompt that discourages echoing and repetition.

    Parameters
    ----------
    query : str
        User question.
    context : str
        Concatenated top-k passages.

    Returns
    -------
    str
        Prompt string.
    """
    return (
        "You are a precise financial assistant.\n"
        "Answer the question in one or two concise sentences using ONLY the context.\n"
        "Do not repeat the question. Do not say 'based on the context'.\n"
        "If the answer is not in the context, say: \"I don't have enough information.\"\n\n"
        f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    )


def _postprocess_answer(answer: str) -> str:
    """
    Clean common LM artifacts like 'Answer:' echoes and repeated lines.

    Parameters
    ----------
    answer : str
        Raw model continuation.

    Returns
    -------
    str
        Cleaned answer string.
    """
    cleaned = re.sub(r"^\s*answer\s*:\s*", "", answer, flags=re.IGNORECASE).strip()
    lines = [ln.strip() for ln in cleaned.splitlines() if ln.strip()]
    uniq = []
    for ln in lines:
        if not uniq or ln != uniq[-1]:
            uniq.append(ln)
    return " ".join(uniq)


def generate_response(query: str,
                      retrieved_docs: list,
                      tokenizer,
                      model,
                      device,
                      max_input_tokens: int = 512,
                      max_new_tokens: int = 80) -> str:
    context = "\n".join([doc["text"][:200] for doc in retrieved_docs])  # trim
    prompt = (
        f"Based on the context, answer the question in one concise sentence.\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\nFinal Answer:"
    )
    enc = tokenizer(prompt, return_tensors="pt",
                    truncation=True,
                    max_length=max_input_tokens).to(device)
    with torch.inference_mode():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # greedy decoding
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    answer = decoded.split("Final Answer:")[-1].strip()
    return answer


# =============================================================================
# Guardrails
# =============================================================================

def validate_query(query: str) -> bool:
    """
    Basic input guardrail for empty or unsafe queries.

    Parameters
    ----------
    query : str
        User question.

    Returns
    -------
    bool
        True if acceptable, else False.
    """
    banned = [r"\bkill\b", r"\battack\b", r"\bbomb\b", r"\bhack\b"]
    if not query or not query.strip():
        return False
    ql = query.lower()
    return not any(re.search(p, ql) for p in banned)


def filter_output(answer: str) -> str:
    """
    Output guardrail: flag obvious hallucination cues and trim length.

    Parameters
    ----------
    answer : str
        Generated answer.

    Returns
    -------
    str
        Safe, trimmed answer or a warning message.
    """
    if not answer:
        return "⚠️ No valid answer generated."
    flags = ["i don’t know", "i don't know", "not available", "fictional"]
    if any(f in answer.lower() for f in flags):
        return "I don't have enough information."
    return answer[:600].strip()

def refine_answer(raw_answer: str, tokenizer, model, device,
                  max_input_tokens: int = 256, max_new_tokens: int = 60) -> str:
    """
    Post-processes the raw generated text by asking the model
    to rewrite it into one clean, concise answer.
    """
    prompt = (
        f"Rewrite the following answer into one clean, concise sentence. "
        f"Remove repetitions and unrelated content.\n\n"
        f"Raw Answer:\n{raw_answer}\n\n"
        f"Refined Answer:"
    )
    enc = tokenizer(prompt, return_tensors="pt",
                    truncation=True,
                    max_length=max_input_tokens).to(device)
    with torch.inference_mode():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    answer = decoded.split("Refined Answer:")[-1].strip()
    return answer

# Load indexes and supporting data
faiss_index = load_faiss_index(FAISS_INDEX_PATH)
bm25_index = load_bm25_index(BM25_INDEX_PATH)
records = load_metadata(META_PATH, CSV_FALLBACK)

# Load embedding model for query -> FAISS
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Load reranker and generator
rr_tok, rr_model, rr_device = build_reranker()
gen_tok, gen_model, gen_device = load_generator("distilgpt2")

# Example query
user_query = "What was Nestlé's cost of goods sold in 2024?"

if validate_query(user_query):
    # Stage 1: Hybrid retrieval
    hybrid = retrieve_hybrid(
        user_query,
        faiss_index,
        bm25_index,
        embedder,
        records,
        top_k=20,
        weight_dense=0.6,
    )

    # Stage 2: Cross-encoder re-ranking
    reranked = rerank_candidates(
        user_query,
        hybrid,
        rr_tok,
        rr_model,
        rr_device,
        top_k=5,
        batch_size=16,
    )

    raw_answer = generate_response(query, final_hits,
                               gen_tok, gen_model, gen_device)
    safe_answer = filter_output(raw_answer)

    # refinement step
    final_answer = refine_answer(safe_answer, gen_tok, gen_model, gen_device)

    print("\n=== Final Answer ===")
    print(final_answer)

else:
    print("⚠️ Query rejected by guardrails.")


=== Final Answer ===
The total number of sales transactions that resulted with Nestlé's customers in Latin America was 12,196 and 11,793 in


In [26]:
# ==========================================================
# 📊 Nestlé RAG Chatbot – Gradio UI (FAISS + BM25, No Chroma)
# ==========================================================
import time
import re
import string
import pickle
from typing import List, Dict, Any, Tuple

import faiss
import numpy as np
import torch
import gradio as gr
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
)

# ----------------------------------------------------------
# Paths
# ----------------------------------------------------------
FAISS_INDEX = (
    "/content/drive/MyDrive/Conversational AI/Processing Files/indexes/nestle_qna.index"
)
META_PKL = (
    "/content/drive/MyDrive/Conversational AI/Processing Files/indexes/metadata.pkl"
)
BM25_PKL = (
    "/content/drive/MyDrive/Conversational AI/Processing Files/indexes/bm25_index.pkl"
)

# ----------------------------------------------------------
# Load indexes and models
# ----------------------------------------------------------
index = faiss.read_index(FAISS_INDEX)

with open(META_PKL, "rb") as f:
    records: List[Dict[str, Any]] = pickle.load(f)

with open(BM25_PKL, "rb") as f:
    bm25 = pickle.load(f)

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

ce_tok = AutoTokenizer.from_pretrained("cross-encoder/ms-marco-MiniLM-L-6-v2")
ce_model = AutoModelForSequenceClassification.from_pretrained(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)
ce_model.eval()

gen_tok = AutoTokenizer.from_pretrained("distilgpt2")
gen_model = AutoModelForCausalLM.from_pretrained("distilgpt2")
gen_model.eval()
if gen_tok.pad_token_id is None:
    gen_tok.pad_token = gen_tok.eos_token

# ----------------------------------------------------------
# Guardrails
# ----------------------------------------------------------
BLOCKED = {"hack", "bomb", "attack", "terror", "violence"}


def is_safe_query(query: str) -> bool:
    """
    Return True if the query is safe to answer.

    Parameters
    ----------
    query : str
        User input.

    Returns
    -------
    bool
        Safety flag.
    """
    return not any(term in query.lower() for term in BLOCKED)


# ----------------------------------------------------------
# Utilities
# ----------------------------------------------------------
def normalize_text(text: str) -> List[str]:
    """
    Lowercase, strip punctuation, and split to tokens.

    Parameters
    ----------
    text : str
        Raw input text.

    Returns
    -------
    List[str]
        Token list for BM25.
    """
    txt = text.lower().translate(str.maketrans("", "", string.punctuation))
    return txt.split()


def rec_text(rec: Dict[str, Any]) -> str:
    """
    Build a displayable text for a record.

    Parameters
    ----------
    rec : Dict[str, Any]
        Metadata record with Question/Answer/document.

    Returns
    -------
    str
        Plain text for retrieval/reranking.
    """
    q = str(rec.get("Question", "")).strip()
    a = str(rec.get("Answer", "")).strip()
    doc = rec.get("document")
    if not doc:
        doc = f"{q} {a}".strip()
    # Remove "Q:"/"A:" prefixes if present
    doc = re.sub(r"^\s*Q:\s*", "", doc, flags=re.IGNORECASE)
    doc = re.sub(r"\s*A:\s*", " ", doc, flags=re.IGNORECASE)
    return doc.strip()


def extract_answer(rec: Dict[str, Any]) -> str:
    """
    Extract the 'Answer' part from a record, cleaning prefixes.

    Parameters
    ----------
    rec : Dict[str, Any]
        Metadata record.

    Returns
    -------
    str
        Clean answer text.
    """
    ans = str(rec.get("Answer", "")).strip()
    if not ans and "document" in rec:
        m = re.search(r"A:\s*(.*)$", rec["document"], flags=re.IGNORECASE | re.DOTALL)
        if m:
            ans = m.group(1).strip()
    ans = re.sub(r"^\s*A:\s*", "", ans, flags=re.IGNORECASE)
    return ans.strip()


def first_sentences(text: str, max_sentences: int = 2) -> str:
    """
    Truncate text to the first one or two sentences.

    Parameters
    ----------
    text : str
        Input text.
    max_sentences : int
        Maximum sentences to keep.

    Returns
    -------
    str
        Truncated text.
    """
    parts = re.split(r"(?<=[.?!])\s+", text.strip())
    return " ".join(parts[:max_sentences]).strip()


# ----------------------------------------------------------
# Retrieval (FAISS dense, BM25 sparse, hybrid merge)
# ----------------------------------------------------------
def retrieve_dense(query: str, top_k: int = 8) -> List[Dict[str, Any]]:
    """
    Dense retrieval with FAISS.

    Parameters
    ----------
    query : str
        User query.
    top_k : int
        Number of results.

    Returns
    -------
    List[Dict[str, Any]]
        Dense hits with scores and indices.
    """
    q_emb = embedder.encode([query]).astype("float32")
    distances, indices = index.search(q_emb, top_k)
    hits = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx < 0:
            continue
        rec = records[idx]
        hits.append(
            {
                "idx": int(idx),
                "text": rec_text(rec),
                "dense_score": float(1.0 / (1.0 + dist)),
                "record": rec,
            }
        )
    return hits


def retrieve_sparse(query: str, top_k: int = 8) -> List[Dict[str, Any]]:
    """
    Sparse retrieval with BM25.

    Parameters
    ----------
    query : str
        User query.
    top_k : int
        Number of results.

    Returns
    -------
    List[Dict[str, Any]]
        Sparse hits with scores and indices.
    """
    tokens = normalize_text(query)
    scores = bm25.get_scores(tokens)
    idxs = np.argsort(scores)[::-1][:top_k]
    hits = []
    for i in idxs:
        rec = records[int(i)]
        hits.append(
            {
                "idx": int(i),
                "text": rec_text(rec),
                "sparse_score": float(scores[int(i)]),
                "record": rec,
            }
        )
    return hits


def hybrid_retrieve(query: str, top_k: int = 8, w_dense: float = 0.6) -> List[Dict[str, Any]]:
    """
    Combine dense and sparse hits into a hybrid score.

    Parameters
    ----------
    query : str
        User query.
    top_k : int
        Number of results to return.
    w_dense : float
        Weight for dense score in [0, 1].

    Returns
    -------
    List[Dict[str, Any]]
        Top-k hybrid hits.
    """
    dense = {h["idx"]: h for h in retrieve_dense(query, top_k)}
    sparse = {h["idx"]: h for h in retrieve_sparse(query, top_k)}

    merged: Dict[int, Dict[str, Any]] = {}
    for d in dense.values():
        merged[d["idx"]] = dict(d)
    for s in sparse.values():
        merged.setdefault(s["idx"], dict(s))
        merged[s["idx"]].update(s)

    for item in merged.values():
        ds = item.get("dense_score", 0.0)
        ss = item.get("sparse_score", 0.0)
        item["hybrid_score"] = w_dense * ds + (1.0 - w_dense) * ss

    ranked = sorted(merged.values(), key=lambda x: x["hybrid_score"], reverse=True)
    return ranked[:top_k]


# ----------------------------------------------------------
# Cross-encoder reranking
# ----------------------------------------------------------
def rerank_cross_encoder(query: str, docs: List[Dict[str, Any]], top_k: int = 3) -> List[Dict[str, Any]]:
    """
    Re-rank candidates with a cross-encoder.

    Parameters
    ----------
    query : str
        User query.
    docs : List[Dict[str, Any]]
        Candidates from hybrid retrieval.
    top_k : int
        Number of final documents.

    Returns
    -------
    List[Dict[str, Any]]
        Top-k reranked documents.
    """
    if not docs:
        return []

    pairs = ([query] * len(docs), [d["text"] for d in docs])
    enc = ce_tok(*pairs, padding=True, truncation=True, return_tensors="pt")

    with torch.no_grad():
        scores = ce_model(**enc).logits.squeeze(-1).tolist()

    for d, s in zip(docs, scores):
        d["rerank_score"] = float(s)

    return sorted(docs, key=lambda x: x["rerank_score"], reverse=True)[:top_k]


# ----------------------------------------------------------
# Answering (extract top answer; LM fallback only if needed)
# ----------------------------------------------------------
def generate_answer(query: str, docs: List[Dict[str, Any]], max_new_tokens: int = 80) -> Tuple[str, str]:
    """
    Produce a concise answer using the top reranked documents.

    Parameters
    ----------
    query : str
        User query.
    docs : List[Dict[str, Any]]
        Reranked documents with records.
    max_new_tokens : int
        Generation cap for fallback.

    Returns
    -------
    Tuple[str, str]
        (final_answer, context_used)
    """
    if not docs:
        return "No relevant information found.", ""

    answers = []
    for d in docs[:3]:
        ans = extract_answer(d["record"])
        if ans:
            answers.append(ans)

    if answers:
        primary = first_sentences(answers[0], max_sentences=2)
        context = "\n".join(f"- {first_sentences(a, 2)}" for a in answers)
        return primary, context

    # Fallback: summarize top texts with a small LM (rarely needed)
    context = "\n".join(first_sentences(d["text"], 2) for d in docs[:3])
    prompt = (
        "Use only the facts below to answer the question concisely.\n"
        f"Facts:\n{context}\n\n"
        f"Question: {query}\n"
        "Answer in one sentence:"
    )
    enc = gen_tok(prompt, return_tensors="pt", truncation=True, max_length=512)
    out = gen_model.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_p=0.9,
        temperature=0.7,
        eos_token_id=gen_tok.eos_token_id,
        pad_token_id=gen_tok.eos_token_id,
    )
    dec = gen_tok.decode(out[0], skip_special_tokens=True)
    ans = dec.split("Answer", 1)[-1].replace(":", " ").strip()
    ans = first_sentences(ans, 2)
    return ans if ans else "No relevant information found.", context


# ----------------------------------------------------------
# End-to-end pipeline
# ----------------------------------------------------------
def rag_pipeline(query: str) -> Tuple[str, str, float, str]:
    """
    Full pipeline: guardrail → retrieval → rerank → answer.

    Parameters
    ----------
    query : str
        User question.

    Returns
    -------
    Tuple[str, str, float, str]
        (answer, context, confidence, elapsed_time)
    """
    if not is_safe_query(query):
        return "❌ Unsafe query detected.", "", 0.0, "N/A"

    t0 = time.time()
    retrieved = hybrid_retrieve(query, top_k=8)
    reranked = rerank_cross_encoder(query, retrieved, top_k=3)
    answer, context = generate_answer(query, reranked)
    elapsed = time.time() - t0

    conf = float(np.mean([d["rerank_score"] for d in reranked])) if reranked else 0.0
    return answer, context, conf, f"{elapsed:.2f}s"


# ----------------------------------------------------------
# Gradio UI
# ----------------------------------------------------------
demo = gr.Interface(
    fn=rag_pipeline,
    inputs=gr.Textbox(label="Ask a financial question about Nestlé"),
    outputs=[
        gr.Textbox(label="Final Answer"),
        gr.Textbox(label="Context Used"),
        gr.Number(label="Confidence (avg rerank score)"),
        gr.Textbox(label="Response Time"),
    ],
    title="📊 Nestlé Financial RAG Assistant (FAISS + BM25)",
    description="Hybrid retrieval (FAISS + BM25), cross-encoder reranking, and robust answer extraction.",
)

demo.launch(debug=True)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://3a81dd48fb34054786.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://3a81dd48fb34054786.gradio.live
